In [4]:
import cv2
import torch
import faiss
import numpy as np
from PIL import Image
from ultralytics import YOLO
from transformers import CLIPProcessor, CLIPModel

d:\Pyongyang-VLM\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
class PersonRetrievalPipeline:
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Using device: {self.device}")

        self.yolo_model = YOLO('yolov8n.pt')
        self.CLIP_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(self.device)
        self.CLIP_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

        self.vector_dim = 1
        # Dùng IndexFlatIP (Inner Product) + L2 Normalize ở bước nhúng vector sẽ tương đương với Cosine Similarity.
        self.index = faiss.IndexFlatIP(self.vector_dim)
        self.metadata = []
        self.clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    
    def apply_CLAHE(self, bgr_img):
        """
        bgr -> LAB
        clahe only L channel (Light) 
        """
        lab = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        cl = self.clahe.apply(l)
        l_img = cv2.merge(cl,a,b)
        return cv2.cvtColor(l_img, cv2.COLOR_LAB2BGR)

    def process_and_index(self, img_path):
        img = cv2.imread(img_path)
        if img is None:
            print("Error, check img path")
            return
        results = self.yolo_model(img, classes=[0]) # 0 mean person in COCO
        crops_pil = []
        bboxes = []

        for result in results:
            for box in result.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                if (x2 - x1) < 20 or (y2 - y1) < 50:
                    continue
                crop_img = img[y1:y2, x1:x2]
                
                # Tiền xử lý: Áp dụng CLAHE cân bằng sáng
                enhanced_crop = self.apply_clahe(crop_img)
                '''BGR to PIL RGB for CLIP'''
                rgb_crop = cv2.cvtColor(enhanced_crop, cv2.COLOR_BGR2RGB)
                pil_img = Image.fromarray(rgb_crop)
                crops_pil.append(pil_img)
                bboxes.append({"file": img_path, "bbox": (x1, y1, x2, y2)})
        if not crops_pil:
            print("Can't find person in the picture")
            
        inputs = self.CLIP_processor(images=crops_pil, return_tensors='pt', padding=True).to(self.device)
        with torch.no_grad():
            image_features = self.clip_model.get_image_features(**inputs)
            # L2 Normalize để chuẩn bị cho Cosine Similarity
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        
        embeddings = image_features.cpu().numpy().astype('float32')
        self.index.add(embeddings)
        self.metadata.extend(bboxes)
        print(f"Đã thêm {len(crops_pil)} người vào database.")

    def search_person(self, text_query, top_k=3):
            """Tìm kiếm người dựa trên văn bản"""
            if self.index.ntotal == 0:
                print("Database trống, vui lòng index ảnh trước.")
                return []

            # Xử lý text query
            inputs = self.processor(text=[text_query], return_tensors="pt", padding=True).to(self.device)
            with torch.no_grad():
                text_features = self.clip_model.get_text_features(**inputs)
                # L2 Normalize
                text_features = text_features / text_features.norm(dim=-1, keepdim=True)
                
            text_embedding = text_features.cpu().numpy().astype('float32')
            
            # Search FAISS
            distances, indices = self.index.search(text_embedding, top_k)
            
            results = []
            for i, idx in enumerate(indices[0]):
                if idx != -1:  # Nếu tìm thấy
                    results.append({
                        "score": float(distances[0][i]),
                        "info": self.metadata[idx]
                    })
            return results

In [9]:
pipeline = PersonRetrievalPipeline()
test_img_path = 'test/4b83aa28958914ed9c6cbb9b8ca2e7d1.jpg'
query = "Find the man wearing a gray shirt and jeans"
pipeline.process_and_index(test_img_path)

Using device: cpu


d:\Pyongyang-VLM\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin.ADMIN-PC\.cache\huggingface\hub\models--openai--clip-vit-base-patch32. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 398/398 [00:00<00:00, 49685.49it/s]
CLIPModel LOAD REPORT from: openai/cli

FileNotFoundError: [Errno 2] No such file or directory: 'test/4b83aa28958914ed9c6cbb9b8ca2e7d1.jpg'

In [ ]:
results = pipeline.search_person(query, top_k=1)
print("Kết quả tìm kiếm:", results)